# Retriever variants summary

Accuracy comparison tables across the four retriever head-to-head sweeps.

Reads `analysis/wandb_runs.csv` (see `scripts/fetch_wandb_results.py` — pulls every
run's logged `accuracy` summary scalar directly from W&B for all exp_ids
below) and builds, for each of the three non-baseline models, a table of
accuracy by retriever x num_shots plus a column averaging each retriever across
shots. A final table aggregates across all models and shots.

Reading from W&B instead of local `results/*.json` matters here: a run can be
"finished" (accuracy logged) on W&B without its local per-sample JSON having
made it to disk, so parsing `results/` undercounts completed runs. Refresh the
CSV first with:
```
/home/aviramom/.conda/envs/multits/bin/python3 scripts/fetch_wandb_results.py
```

Retriever groups, each tied to the exp_id of the script that produced it:
- `0-shot` / `random` <- `retriever_comparison_v1` (`scripts/submit_tse_exp.sh`)
- `ts_compress` / `ts_multivec` / `ts_windowagg` <- `ts_encoder_variants_v2` (`scripts/submit_tse_ts_variants.sh`)
- `vision_ts` / `delay_dino` <- `dino3_vision_variants_v1` (`scripts/submit_tse_dino3_variants.sh`)
- `text` / `text_bge` <- `text_encoder_variants_v1` (`scripts/submit_tse_text_variants.sh`)
- `spectral` / `stats` / `vision_wavelet` <- `freqstat_variants_v1` (`scripts/submit_tse_freqstat_variants.sh`)

The second block below appends more groups (see that section's markdown):
- `top3 MR-RRF` <- `top3_mrrf_k{10,60,100}_v1` (`scripts/submit_tse_top3_mrrf_variants.sh`)
- `dtw` <- `dtw_v1` (`scripts/submit_tse_dtw.sh`)
- `twostage-delay_dino-text` / `twostage-vision_wavelet-text` <- `twostage_v1` (`scripts/submit_tse_twostage.sh`)
- `siglip_plot` / `siglip_delay` <- `siglip_variants_v1` (`scripts/submit_tse_siglip_variants.sh`)

In [ ]:
import os

import pandas as pd
from IPython.display import Markdown, display

WANDB_CSV = os.path.join(os.getcwd(), "wandb_runs.csv")
SHOTS_KSHOT = [1, 2, 3, 5, 8]

MODELS = [
    ("Qwen/Qwen3-VL-8B-Instruct", "Qwen3-VL-8B"),
    ("bytedance-research/ChatTS-8B", "ChatTS-8B"),
    ("Qwen/Qwen3-8B-vllm", "Qwen3-8B (vLLM)"),
]

# (retriever_id, exp_id, display label, group label)
RETRIEVER_SPECS = [
    ("none", "retriever_comparison_v1", "0-shot (no retrieval)", "Baseline"),
    ("random", "retriever_comparison_v1", "random", "Baseline"),
    ("ts_compress", "ts_encoder_variants_v2", "ts_compress", "MOMENT (ts_variants)"),
    ("ts_multivec", "ts_encoder_variants_v2", "ts_multivec", "MOMENT (ts_variants)"),
    ("ts_windowagg", "ts_encoder_variants_v2", "ts_windowagg", "MOMENT (ts_variants)"),
    ("vision_ts", "dino3_vision_variants_v1", "vision_ts (plot embed)", "Vision (dino3_variants)"),
    ("delay_dino", "dino3_vision_variants_v1", "delay_dino (delay embed)", "Vision (dino3_variants)"),
    ("text", "text_encoder_variants_v1", "text (MiniLM)", "Text (text_variants)"),
    ("text_bge", "text_encoder_variants_v1", "text_bge (BGE-large)", "Text (text_variants)"),
    ("spectral", "freqstat_variants_v1", "spectral", "Freq/Stats (freqstat_variants)"),
    ("stats", "freqstat_variants_v1", "stats", "Freq/Stats (freqstat_variants)"),
    ("vision_wavelet", "freqstat_variants_v1", "vision_wavelet", "Freq/Stats (freqstat_variants)"),
]

# Top-3 MR-RRF fusion (vision_ts + delay_dino + vision_wavelet, the best retriever
# from each sweep above) swept over 3 RRF smoothing constants — one exp_id per k
# (see scripts/submit_tse_top3_mrrf_variants.sh). Jobs haven't run yet as of this
# writing, so these rows show "—" until the CSV is refreshed after they finish.
TOP3_MRRF_RETRIEVER = "rrf-vision_ts-delay_dino-vision_wavelet"
NEW_RETRIEVER_SPECS = [
    (TOP3_MRRF_RETRIEVER, "top3_mrrf_k10_v1", "top3 MR-RRF (rrf_k=10)", "Top-3 MR-RRF (rrf_k sweep)"),
    (TOP3_MRRF_RETRIEVER, "top3_mrrf_k60_v1", "top3 MR-RRF (rrf_k=60)", "Top-3 MR-RRF (rrf_k sweep)"),
    (TOP3_MRRF_RETRIEVER, "top3_mrrf_k100_v1", "top3 MR-RRF (rrf_k=100)", "Top-3 MR-RRF (rrf_k sweep)"),
    # DTW shape-similarity baseline: raw-series z-scored signatures compared by
    # Sakoe-Chiba-banded DTW (tslearn, CPU) — see scripts/submit_tse_dtw.sh.
    ("dtw", "dtw_v1", "dtw (shape / tslearn)", "DTW baseline"),
    # Two-stage coarse-to-fine: pull a coarse TS-similar candidate set
    # (--stage1_candidates=50) then re-rank by text similarity of the question —
    # see scripts/submit_tse_twostage.sh. Both variants differ only in the stage-1
    # TS encoder (DINOv3 delay-embedding vs. CWT scalogram).
    ("twostage-delay_dino-text", "twostage_v1", "two-stage delay_dino→text", "Two-stage (coarse→fine)"),
    ("twostage-vision_wavelet-text", "twostage_v1", "two-stage vision_wavelet→text", "Two-stage (coarse→fine)"),
    # SigLIP2 fused image+text retrievers: series image and question text encoded
    # by the same SigLIP2 towers into one shared space, pooled into a single fused
    # vector per item — see scripts/submit_tse_siglip_variants.sh. The plot/delay
    # pair mirrors the DINOv3 vision_ts/delay_dino pair (same image renderings),
    # isolating the encoder + text fusion as the variable.
    ("siglip_plot", "siglip_variants_v1", "siglip_plot (plot + text)", "SigLIP2 fused (siglip_variants)"),
    ("siglip_delay", "siglip_variants_v1", "siglip_delay (delay embed + text)", "SigLIP2 fused (siglip_variants)"),
]

EXTENDED_RETRIEVER_SPECS = RETRIEVER_SPECS + NEW_RETRIEVER_SPECS

In [2]:
def load_runs() -> pd.DataFrame:
    df = pd.read_csv(WANDB_CSV)
    df = df.rename(columns={"random_seed": "seed", "accuracy": "acc"})
    return df[["exp_id", "method", "retriever", "num_shots", "seed", "acc"]]


df = load_runs()
df.head()

,exp_id,method,retriever,num_shots,seed,acc
0,retriever_comparison_v1,Qwen/Qwen3-8B-vllm,delay_dino,1,0,0.487936
1,retriever_comparison_v1,Qwen/Qwen3-8B-vllm,delay_dino,1,1,0.490617
2,retriever_comparison_v1,Qwen/Qwen3-8B-vllm,delay_dino,1,2021,0.487936
3,dino3_vision_variants_v1,Qwen/Qwen3-8B-vllm,delay_dino,1,2021,0.513405
4,retriever_comparison_v1,Qwen/Qwen3-8B-vllm,delay_dino,2,0,0.536193


In [3]:
def cell(df: pd.DataFrame, exp_id: str, method: str, retriever: str, shot: int):
    """Mean accuracy across seeds for one (exp_id, method, retriever, shot); None if absent."""
    sub = df[
        (df.exp_id == exp_id)
        & (df.method == method)
        & (df.retriever == retriever)
        & (df.num_shots == shot)
    ]
    if sub.empty:
        return None
    return sub["acc"].mean()

In [4]:
def per_model_df(df: pd.DataFrame, method: str, specs=RETRIEVER_SPECS) -> pd.DataFrame:
    """Retriever x num_shots accuracy table for one model, as a real DataFrame
    (Group, Retriever) MultiIndex rows x (0-shot, k=1..8, Avg) columns."""
    rows = []
    for retriever, exp_id, label, group in specs:
        zero = cell(df, exp_id, method, retriever, 0) if retriever == "none" else None
        kshot_vals = (
            [None] * len(SHOTS_KSHOT)
            if retriever == "none"
            else [cell(df, exp_id, method, retriever, s) for s in SHOTS_KSHOT]
        )
        present = [v for v in kshot_vals if v is not None]
        avg = sum(present) / len(present) if present else None

        row = {"Group": group, "Retriever": label, "0-shot": zero}
        for s, v in zip(SHOTS_KSHOT, kshot_vals):
            row[f"k={s}"] = v
        row["Avg (k-shot)"] = avg
        rows.append(row)

    return pd.DataFrame(rows).set_index(["Group", "Retriever"])


def show_table(table: pd.DataFrame, title: str):
    display(Markdown(f"### {title}"))
    display(
        table.style.format("{:.3f}", na_rep="—")
        .background_gradient(cmap="Greens", axis=None)
        .highlight_max(axis=0, props="font-weight: bold; background-color: gold; color: black;")
    )

In [5]:
for method, label in MODELS:
    show_table(per_model_df(df, method), label)

### Qwen3-VL-8B

### ChatTS-8B

### Qwen3-8B (vLLM)

In [6]:
def summary_df(df: pd.DataFrame, specs=RETRIEVER_SPECS) -> pd.DataFrame:
    """Overall avg accuracy + coverage per retriever, aggregated across all models and shots."""
    rows = []
    for retriever, exp_id, label, group in specs:
        shots = [0] if retriever == "none" else SHOTS_KSHOT
        expected = len(MODELS) * len(shots)
        vals = [
            v
            for method, _ in MODELS
            for s in shots
            if (v := cell(df, exp_id, method, retriever, s)) is not None
        ]
        avg = sum(vals) / len(vals) if vals else None
        rows.append(
            {
                "Retriever": label,
                "Group": group,
                "Overall avg accuracy": avg,
                "Coverage": f"{len(vals)}/{expected}",
            }
        )
    return pd.DataFrame(rows).set_index(["Group", "Retriever"])


display(Markdown("### Overall summary (aggregated across all models and shots)"))
display(
    summary_df(df)
    .style.format({"Overall avg accuracy": "{:.3f}"}, na_rep="—")
    .background_gradient(cmap="Greens", subset=["Overall avg accuracy"])
    .highlight_max(
        axis=0,
        subset=["Overall avg accuracy"],
        props="font-weight: bold; background-color: gold; color: black;",
    )
)

### Overall summary (aggregated across all models and shots)

## Same tables, with the top-3 MR-RRF fusion + DTW + two-stage + SigLIP2 retrievers added

Four extra groups are appended below:

- **Top-3 MR-RRF** — `vision_ts` + `delay_dino` + `vision_wavelet` (the best retriever
  from each sweep above), fused via Reciprocal Rank Fusion and swept over 3 RRF smoothing
  constants (`rrf_k` in {10, 60, 100} — see `scripts/submit_tse_top3_mrrf_variants.sh`).
- **DTW baseline** — raw-series shape similarity via Sakoe-Chiba-banded DTW (`tslearn`,
  CPU; exp_id `dtw_v1` — see `scripts/submit_tse_dtw.sh`).
- **Two-stage (coarse→fine)** — pull a coarse TS-similar candidate set
  (`--stage1_candidates=50`) then re-rank by text similarity of the question; the two
  variants differ only in the stage-1 TS encoder (DINOv3 delay-embedding vs. CWT
  scalogram; exp_id `twostage_v1` — see `scripts/submit_tse_twostage.sh`).
- **SigLIP2 fused (image+text)** — the series image and the question text are encoded
  by the same SigLIP2 encoder (`google/siglip2-base-patch16-224`, shared image/text
  space) and pooled into one fused vector per item; the two variants differ only in
  the image rendering (`siglip_plot` = vision_ts line plot, `siglip_delay` =
  delay_dino delay-embedding image), so vs. the DINOv3 pair they isolate the
  encoder + text fusion (exp_id `siglip_variants_v1` — see
  `scripts/submit_tse_siglip_variants.sh`).

Any group whose jobs haven't run yet shows "—" until the CSV is refreshed after they finish.

In [7]:
for method, label in MODELS:
    show_table(per_model_df(df, method, specs=EXTENDED_RETRIEVER_SPECS), f"{label} (incl. top-3 MR-RRF)")

### Qwen3-VL-8B (incl. top-3 MR-RRF)

### ChatTS-8B (incl. top-3 MR-RRF)

### Qwen3-8B (vLLM) (incl. top-3 MR-RRF)

In [8]:
display(Markdown("### Overall summary (incl. top-3 MR-RRF)"))
display(
    summary_df(df, specs=EXTENDED_RETRIEVER_SPECS)
    .style.format({"Overall avg accuracy": "{:.3f}"}, na_rep="—")
    .background_gradient(cmap="Greens", subset=["Overall avg accuracy"])
    .highlight_max(
        axis=0,
        subset=["Overall avg accuracy"],
        props="font-weight: bold; background-color: gold; color: black;",
    )
)

### Overall summary (incl. top-3 MR-RRF)